# 01 — Explorar Data V1.0

**Checkpoint: Día 1 — revisión inicial.** Objetivo: entender el esquema real de las 3 capas antes de tocar código de ingesta. Todo lo que se ejecuta aquí es de solo lectura sobre `../../data/raw/ (config.DATA_ROOT)`.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from saberlink import config, schema

pd.set_option('display.max_colwidth', 80)

## Manifiestos declarados vs. conteos reales
Cada capa trae su propio `dataset_manifest*.json` con `record_counts`. Verificamos que coincidan con lo que de verdad hay en los CSV.

In [2]:
import json

manifests = [
    config.INSTITUTION_DIR / 'dataset_manifest.json',
    config.PEOPLE_DIR / 'dataset_manifest_block_B.json',
    config.NEEDS_DIR / 'dataset_manifest_block_C.json',
]
for path in manifests:
    data = json.loads(path.read_text(encoding='utf-8'))
    print(path.name, '->', data.get('record_counts'))

dataset_manifest.json -> {'faculties': 6, 'programs': 18, 'research_groups': 24, 'research_lines': 60, 'institutional_capabilities': 96, 'source_catalog': 35}
dataset_manifest_block_B.json -> {'researchers': 180, 'researcher_expertise': 720, 'subjects': 126, 'competencies': 252, 'learning_outcomes': 378, 'researcher_group': 180}
dataset_manifest_block_C.json -> {'institutional_needs': 42, 'projects': 320, 'theses': 650, 'publications': 360, 'researcher_project': 746, 'project_group': 320, 'thesis_advisor': 650, 'publication_researcher': 720, 'publication_project': 200, 'documents': 60}


In [3]:
for spec in schema.ENTITY_SPECS.values():
    df = pd.read_csv(config.DATA_ROOT / spec.relative_path, encoding='utf-8-sig', dtype=str)
    print(f"{spec.entity_type:4s}  {spec.relative_path:55s}  {len(df):4d} filas  cols={list(df.columns)}")

FAC   01_institution/faculties.csv                                6 filas  cols=['faculty_id', 'faculty_name', 'description', 'strategic_focus', 'active']
PRG   01_institution/programs.csv                                18 filas  cols=['program_id', 'faculty_id', 'program_name', 'academic_level', 'description', 'disciplinary_area', 'graduate_profile', 'strategic_topics', 'active']
GRP   01_institution/research_groups.csv                         24 filas  cols=['group_id', 'group_name', 'faculty_id', 'description', 'mission', 'main_area', 'interdisciplinary', 'creation_year', 'status']
LIN   01_institution/research_lines.csv                          60 filas  cols=['line_id', 'group_id', 'line_name', 'description', 'keywords', 'active']
CAP   01_institution/institutional_capabilities.csv              96 filas  cols=['capability_id', 'capability_name', 'capability_type', 'description', 'responsible_unit', 'available_resources', 'application_domains', 'maturity_level', 'status']
SRC   01_

## BOM en los encabezados
Todos los CSV traen BOM (`\xef\xbb\xbf`) delante del primer encabezado. Si se lee sin `encoding='utf-8-sig'`, la primera columna queda como `﻿need_id` y rompe cualquier join por ID.

In [4]:
raw_bytes = (config.NEEDS_DIR / 'institutional_needs.csv').read_bytes()[:6]
print('primeros bytes:', raw_bytes)
print('con utf-8-sig, header limpio:', pd.read_csv(config.NEEDS_DIR / 'institutional_needs.csv', encoding='utf-8-sig', nrows=0).columns.tolist())

primeros bytes: b'\xef\xbb\xbfnee'
con utf-8-sig, header limpio: ['need_id', 'title', 'description', 'originating_unit', 'context', 'expected_impact', 'priority', 'year', 'status']


## Hallazgo: dos plantillas distintas en `institutional_needs.csv`

NEED-001..020 usan una plantilla rica con términos de dominio embebidos en `description`. NEED-021..042 usan una plantilla administrativa genérica donde `description`/`context`/`expected_impact` son casi idénticos entre sí y solo el `title` diferencia. Esto importa para el diseño del scoring (ver `03_scoring_demo.ipynb`): la similitud semántica cruda no basta para diferenciar dentro de la familia genérica sin incluir siempre `title`.

In [5]:
needs = pd.read_csv(config.NEEDS_DIR / 'institutional_needs.csv', encoding='utf-8-sig')
print('--- familia rica (NEED-001) ---')
print(needs.loc[needs.need_id=='NEED-001', 'description'].iloc[0])
print()
print('--- familia genérica (NEED-021) ---')
print(needs.loc[needs.need_id=='NEED-021', 'description'].iloc[0])
print()
print('--- familia genérica (NEED-022), casi idéntica salvo el título ---')
print(needs.loc[needs.need_id=='NEED-022', 'description'].iloc[0])

--- familia rica (NEED-001) ---
La institución requiere fortalecer su capacidad para abordar predicción y prevención de deserción estudiantil mediante el aprovechamiento articulado de información, conocimiento previo y capacidades existentes. El problema se expresa en términos de permanencia estudiantil, riesgo académico y trayectorias educativas, sin prescribir una solución tecnológica específica.

--- familia genérica (NEED-021) ---
La institución requiere consolidar información distribuida para actualización de portafolio de investigación aplicada y sustentar decisiones con evidencia verificable.

--- familia genérica (NEED-022), casi idéntica salvo el título ---
La institución requiere consolidar información distribuida para mapa de capacidades para cooperación internacional y sustentar decisiones con evidencia verificable.


## Hallazgo: `research_lines.csv.keywords` usa `;` como separador de palabras

A diferencia de todos los demás archivos (donde `;` separa frases completas, ej. `"optimización; procesamiento de señales"`), aquí `;` separa palabras sueltas de UNA sola frase: `"aprendizaje;automático"` significa "aprendizaje automático", no dos términos distintos. Tratarlo como lista normal inyecta basura ("el", "en", "y") al vocabulario de dominio — corregido en `saberlink/schema.py` vía `space_joined_fields`.

In [6]:
lines = pd.read_csv(config.INSTITUTION_DIR / 'research_lines.csv', encoding='utf-8-sig')
lines[['line_id', 'line_name', 'keywords']].head(6)

,line_id,line_name,keywords
0,LIN-001,Aprendizaje automático,aprendizaje;automático
1,LIN-002,Sistemas inteligentes,sistemas;inteligentes
2,LIN-003,Inteligencia artificial explicable,inteligencia;artificial;explicable
3,LIN-004,Arquitecturas de software,arquitecturas;de;software
4,LIN-005,Ingeniería de software basada en evidencia,ingeniería;de;software;basada;en;evidencia
5,LIN-006,Sistemas distribuidos,sistemas;distribuidos


## Campos de texto libre (unidad de embedding) por tipo de entidad
Estos son los campos que se embeben POR CAMPO, no por entidad completa (`saberlink/ingest.py::build_fields_index`) — así la evidencia puede citar el campo exacto, no todo el registro.

In [7]:
for etype, spec in schema.ENTITY_SPECS.items():
    if spec.text_fields:
        print(f'{etype:4s}: {spec.text_fields}')

FAC : ('description', 'strategic_focus')
PRG : ('description', 'graduate_profile', 'strategic_topics')
GRP : ('group_name', 'description', 'mission')
LIN : ('description',)
CAP : ('capability_name', 'description')
INV : ('academic_background', 'profile_summary', 'research_interests', 'methodological_expertise', 'application_domains')
EXP : ('expertise_name',)
SUB : ('description', 'purpose', 'main_topics')
COM : ('description',)
LO  : ('outcome_description',)
NEED: ('title', 'description', 'context', 'expected_impact')
PRJ : ('title', 'problem_statement', 'abstract', 'general_objective', 'methodology', 'expected_results', 'application_context')
THS : ('title', 'abstract', 'problem_statement', 'general_objective', 'methodology', 'main_results', 'conclusions')
PUB : ('title', 'abstract')
